# Groundwater Data Engineering for Maharashtra Pune

This notebook performs an end-to-end PySpark workflow on the raw national groundwater dataset and builds a high-quality regional dataset for Maharashtra Pune.

Pipeline steps:
1. Initial data exploration
2. District and coordinate density analysis
3. Station sample survival analysis
4. Pune bounding box filtering
5. Cutoff optimization between 100 and 110 samples
6. Extraction and cleaning of the selected station records
7. Final quality checks and export to CSV


In [1]:
from pyspark.sql import SparkSession, functions as F
import os
import shutil

spark = (
    SparkSession.builder
    .appName("BhujalMitra-Groundwater-Engineering")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

RAW_PATH = "data/Groundwater India Data.csv"
OUTPUT_PATH = "data/Maharashtra_Pune_Filtered.csv"
TEMP_DIR = "data/.tmp_pune_filtered_output"

PUNE_BBOX = {
    "lat_min": 15.5,
    "lat_max": 21.0,
    "lon_min": 72.5,
    "lon_max": 77.0
}
OPTIMAL_CUTOFF = 106


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/29 00:44:26 WARN Utils: Your hostname, mridulUbuntu, resolves to a loopback address: 127.0.1.1; using 10.51.26.78 instead (on interface wlp2s0)
26/03/29 00:44:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/03/29 00:44:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATH)
)

row_count = raw_df.count()
column_count = len(raw_df.columns)
print(f"Row count: {row_count}")
print(f"Column count: {column_count}")
print("Columns:", raw_df.columns)

raw_df.printSchema()
raw_df.show(5, truncate=False)


Row count: 416952
Column count: 15
Columns: ['time_idx', 'station_id', 'datetime', 'target', 'rainfall', 't  kmnpnn]2m', 't2m_max', 't2m_min', 'month', 'latitude', 'longitude', 'wellDepth', 'wellAquiferType_encoded', 'State_encoded', 'District_encoded']
root
 |-- time_idx: integer (nullable = true)
 |-- station_id: string (nullable = true)
 |-- datetime: date (nullable = true)
 |-- target: double (nullable = true)
 |-- rainfall: double (nullable = true)
 |-- t  kmnpnn]2m: double (nullable = true)
 |-- t2m_max: double (nullable = true)
 |-- t2m_min: double (nullable = true)
 |-- month: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- wellDepth: double (nullable = true)
 |-- wellAquiferType_encoded: integer (nullable = true)
 |-- State_encoded: integer (nullable = true)
 |-- District_encoded: integer (nullable = true)



+--------+----------+----------+--------+--------+------------+-------+-------+-----+--------+---------+---------+-----------------------+-------------+----------------+
|time_idx|station_id|datetime  |target  |rainfall|t  kmnpnn]2m|t2m_max|t2m_min|month|latitude|longitude|wellDepth|wellAquiferType_encoded|State_encoded|District_encoded|
+--------+----------+----------+--------+--------+------------+-------+-------+-----+--------+---------+---------+-----------------------+-------------+----------------+
|0       |CGWHYD0426|2023-07-14|-11.305 |0.02    |30.62       |36.39  |24.98  |7    |14.9083 |78.0083  |31.0     |0                      |0            |0               |
|1       |CGWHYD0426|2023-07-15|-11.2775|4.11    |30.44       |36.97  |25.01  |7    |14.9083 |78.0083  |31.0     |0                      |0            |0               |
|2       |CGWHYD0426|2023-07-16|-11.2675|0.09    |29.04       |34.03  |24.77  |7    |14.9083 |78.0083  |31.0     |0                      |0           

In [3]:
district_distribution = raw_df.groupBy("District_encoded").count().orderBy(F.desc("count"))
district_distribution.show(10, truncate=False)

density_grid = (
    raw_df
    .withColumn("lat_bin", F.round(F.col("latitude"), 1))
    .withColumn("lon_bin", F.round(F.col("longitude"), 1))
    .groupBy("lat_bin", "lon_bin")
    .count()
    .orderBy(F.desc("count"))
)
density_grid.show(15, truncate=False)

regional_windows = (
    raw_df
    .withColumn(
        "region_window",
        F.when(
            F.col("latitude").between(15.0, 22.5) & F.col("longitude").between(72.0, 81.0),
            F.lit("Maharashtra_window")
        )
        .when(
            F.col("latitude").between(17.0, 22.5) & F.col("longitude").between(81.0, 87.5),
            F.lit("Odisha_window")
        )
        .otherwise(F.lit("Other"))
    )
    .groupBy("region_window")
    .count()
    .orderBy(F.desc("count"))
)
regional_windows.show(truncate=False)


+----------------+------+
|District_encoded|count |
+----------------+------+
|1               |415695|
|0               |1257  |
+----------------+------+



+-------+-------+-----+
|lat_bin|lon_bin|count|
+-------+-------+-----+
|30.4   |77.2   |925  |
|20.3   |85.8   |814  |
|22.7   |75.6   |769  |
|9.7    |76.6   |643  |
|20.0   |86.2   |642  |
|30.9   |75.5   |636  |
|22.0   |81.5   |630  |
|11.7   |76.0   |613  |
|20.9   |85.8   |587  |
|20.6   |86.6   |567  |
|23.0   |76.7   |563  |
|20.3   |82.8   |552  |
|23.9   |76.9   |539  |
|23.0   |80.4   |538  |
|29.2   |77.0   |532  |
+-------+-------+-----+
only showing top 15 rows


+------------------+------+
|region_window     |count |
+------------------+------+
|Other             |284473|
|Maharashtra_window|88655 |
|Odisha_window     |43824 |
+------------------+------+



In [4]:
station_sample_counts = raw_df.groupBy("station_id").count().cache()
print(f"Total stations in raw dataset: {station_sample_counts.count()}")

thresholds = [100, 200, 300, 350, 360, 370, 380, 390]
survival_rows = []
for t in thresholds:
    surviving = station_sample_counts.filter(F.col("count") >= t).count()
    survival_rows.append((t, surviving))

survival_df = spark.createDataFrame(survival_rows, ["minimum_samples", "stations_remaining"])
survival_df.orderBy("minimum_samples").show(truncate=False)

elite_380 = station_sample_counts.filter(F.col("count") >= 380).count()
print(f"Elite stations with at least 380 samples: {elite_380}")


Total stations in raw dataset: 6440


+---------------+------------------+
|minimum_samples|stations_remaining|
+---------------+------------------+
|100            |1444              |
|200            |405               |
|300            |126               |
|350            |98                |
|360            |86                |
|370            |70                |
|380            |55                |
|390            |30                |
+---------------+------------------+



Elite stations with at least 380 samples: 55


In [5]:
pune_df = raw_df.filter(
    F.col("latitude").between(PUNE_BBOX["lat_min"], PUNE_BBOX["lat_max"]) &
    F.col("longitude").between(PUNE_BBOX["lon_min"], PUNE_BBOX["lon_max"])
)

print(f"Pune-region row count: {pune_df.count()}")
print(f"Pune-region station count: {pune_df.select('station_id').distinct().count()}")

pune_station_counts = pune_df.groupBy("station_id").count().orderBy(F.desc("count"))
pune_station_counts.show(10, truncate=False)


Pune-region row count: 28572


Pune-region station count: 389


+----------+-----+
|station_id|count|
+----------+-----+
|CGWBNG0549|409  |
|CGWBNG0541|402  |
|CGWBNG0596|317  |
|CGWBNG0579|305  |
|CGWJPR0202|297  |
|CGWBNG0583|295  |
|CGWBNG0587|292  |
|CGWJPR0246|286  |
|CGWBNG0555|281  |
|CGWNAG2007|276  |
+----------+-----+
only showing top 10 rows


In [6]:
cutoff_scan_rows = []
for cutoff in range(100, 111):
    remaining = pune_station_counts.filter(F.col("count") >= cutoff).count()
    cutoff_scan_rows.append((cutoff, remaining))

cutoff_scan_df = spark.createDataFrame(cutoff_scan_rows, ["cutoff", "stations_remaining"]).orderBy("cutoff")
cutoff_scan_df.show(20, truncate=False)

stations_at_optimal_cutoff = pune_station_counts.filter(F.col("count") >= OPTIMAL_CUTOFF).count()
print(f"Selected cutoff: {OPTIMAL_CUTOFF}")
print(f"Stations retained at cutoff {OPTIMAL_CUTOFF}: {stations_at_optimal_cutoff}")


+------+------------------+
|cutoff|stations_remaining|
+------+------------------+
|100   |128               |
|101   |123               |
|102   |120               |
|103   |115               |
|104   |109               |
|105   |104               |
|106   |94                |
|107   |88                |
|108   |81                |
|109   |70                |
|110   |62                |
+------+------------------+



Selected cutoff: 106
Stations retained at cutoff 106: 94


In [7]:
selected_station_ids = pune_station_counts.filter(F.col("count") >= OPTIMAL_CUTOFF).select("station_id")
filtered_df = pune_df.join(selected_station_ids, on="station_id", how="inner")

corrupted_temp_col = next((c for c in raw_df.columns if "kmnpnn" in c), "t  kmnpnn]2m")

final_df = (
    filtered_df
    .drop("State_encoded", "District_encoded")
    .withColumnRenamed("target", "groundwater_level_target")
    .withColumnRenamed(corrupted_temp_col, "t2m_avg")
)

print(f"Final filtered row count: {final_df.count()}")
print(f"Final filtered station count: {final_df.select('station_id').distinct().count()}")
print("Final columns:")
print(final_df.columns)

final_df.show(5, truncate=False)


Final filtered row count: 14641


Final filtered station count: 94
Final columns:
['station_id', 'time_idx', 'datetime', 'groundwater_level_target', 'rainfall', 't2m_avg', 't2m_max', 't2m_min', 'month', 'latitude', 'longitude', 'wellDepth', 'wellAquiferType_encoded']


+----------+--------+----------+------------------------+--------+-------+-------+-------+-----+--------+---------+---------+-----------------------+
|station_id|time_idx|datetime  |groundwater_level_target|rainfall|t2m_avg|t2m_max|t2m_min|month|latitude|longitude|wellDepth|wellAquiferType_encoded|
+----------+--------+----------+------------------------+--------+-------+-------+-------+-----+--------+---------+---------+-----------------------+
|CGWNAG2007|0       |2024-05-17|-6.21                   |6.87    |31.17  |33.92  |28.9   |5    |15.5778 |73.9181  |90.0     |1                      |
|CGWNAG2007|1       |2024-05-18|-6.2125                 |0.5     |31.21  |33.79  |29.28  |5    |15.5778 |73.9181  |90.0     |1                      |
|CGWNAG2007|2       |2024-05-19|-6.1                    |11.02   |30.18  |32.15  |28.93  |5    |15.5778 |73.9181  |90.0     |1                      |
|CGWNAG2007|3       |2024-05-20|-5.795                  |14.22   |30.2   |33.14  |28.26  |5    |15.5

In [8]:
if os.path.exists(TEMP_DIR):
    shutil.rmtree(TEMP_DIR)

final_df.coalesce(1).write.mode("overwrite").option("header", True).csv(TEMP_DIR)

part_files = [name for name in os.listdir(TEMP_DIR) if name.startswith("part-") and name.endswith(".csv")]
if not part_files:
    raise RuntimeError("No CSV part file found in temporary output directory")

if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)

shutil.move(os.path.join(TEMP_DIR, part_files[0]), OUTPUT_PATH)
shutil.rmtree(TEMP_DIR)

print(f"Saved filtered dataset to: {OUTPUT_PATH}")


Saved filtered dataset to: data/Maharashtra_Pune_Filtered.csv


In [9]:
missing_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in final_df.columns]
missing_summary = final_df.select(*missing_exprs)
missing_summary.show(truncate=False)

duplicate_rows = final_df.count() - final_df.dropDuplicates().count()
print(f"Duplicate row count: {duplicate_rows}")

print("Final schema:")
final_df.printSchema()

print(f"Validation row count: {final_df.count()}")
print(f"Validation station count: {final_df.select('station_id').distinct().count()}")


+----------+--------+--------+------------------------+--------+-------+-------+-------+-----+--------+---------+---------+-----------------------+
|station_id|time_idx|datetime|groundwater_level_target|rainfall|t2m_avg|t2m_max|t2m_min|month|latitude|longitude|wellDepth|wellAquiferType_encoded|
+----------+--------+--------+------------------------+--------+-------+-------+-------+-----+--------+---------+---------+-----------------------+
|0         |0       |0       |0                       |0       |0      |0      |0      |0    |0       |0        |0        |0                      |
+----------+--------+--------+------------------------+--------+-------+-------+-------+-----+--------+---------+---------+-----------------------+



Duplicate row count: 0
Final schema:
root
 |-- station_id: string (nullable = true)
 |-- time_idx: integer (nullable = true)
 |-- datetime: date (nullable = true)
 |-- groundwater_level_target: double (nullable = true)
 |-- rainfall: double (nullable = true)
 |-- t2m_avg: double (nullable = true)
 |-- t2m_max: double (nullable = true)
 |-- t2m_min: double (nullable = true)
 |-- month: integer (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- wellDepth: double (nullable = true)
 |-- wellAquiferType_encoded: integer (nullable = true)



Validation row count: 14641


Validation station count: 94


In [10]:
spark.stop()
print("Spark session stopped.")


Spark session stopped.
